##### Importando bibliotecas nativas

In [ ]:
import pandas as pd
import numpy as np

##### Lendo dados do arquivo de origem

In [ ]:
df_original = pd.read_csv('..\\Origem\\nintendo-sending-packages.csv',sep=";")

##### Colunas, total de itens não nulos e tipos de dados na tabela original

In [ ]:
df_original.info()

##### Estatística na tabela de origem

In [ ]:
df_original.describe()

##### Copiando os registros para uma nova tabela, onde faremos a limpeza e tratamento dos dados

In [ ]:
df_nintendo = df_original.copy()
df_nintendo

##### Alterando o tipo da coluna 'Send Date', de texto para data.

In [ ]:
df_nintendo['Send Date'] = pd.to_datetime(df_nintendo['Send Date'], dayfirst=True, errors='coerce')

##### Renomeando um jogo

In [ ]:
df_nintendo['Game'] = df_nintendo['Game'].str.replace('The Witcher 3: Wild Hunt - Complete Edition - Switch',   
                                                      'The Witcher 3: Wild Hunt (Complete Edition) - Switch',   
                                                      regex=False)

##### Separando o título do jogo e o tipo do console

In [ ]:
jogoCompleto = df_nintendo['Game'].str.split(' - ', n=1, expand=True)

df_nintendo['Game'] = jogoCompleto[0].fillna('Não disponível')
df_nintendo['Console'] = jogoCompleto[1].fillna('Não disponível')

display(df_nintendo)

##### Vamos padronizar os valores monetários, removendo os $ 

In [ ]:
df_nintendo['Preço Unit Price'] = df_nintendo['Preço Unit Price'].str.replace('$','',regex=False)

In [ ]:
df_nintendo.info() # Colunas, total de itens não nulos e tipos de dados em df_nintendo

In [ ]:
df_nintendo.head(12) # As 12 primeiras linhas

##### Convertendo os valores numéricos (para que deixem de ser colunas de texto)

In [ ]:
colunas_numericas = ['Stock Quantity', 'Preço Unit Price']
df_nintendo[colunas_numericas] = df_nintendo[colunas_numericas].apply(pd.to_numeric, errors='coerce')

In [ ]:
# Convertendo coluna float para int
df_nintendo['Stock Quantity'] = df_nintendo['Stock Quantity'].astype('Int64')

##### Vamos criar a coluna que vai receber o produto de Stock Quantity e Preço Unit Price

In [ ]:
def calcular_total(row):
    if pd.isna(row['Stock Quantity']) or pd.isna(row['Preço Unit Price']):
        return np.nan
    return round(row['Stock Quantity'] * row['Preço Unit Price'], 2)

df_nintendo['valor_total_estoque'] = df_nintendo.apply(calcular_total, axis=1)

In [ ]:
df_nintendo.info()

##### Renomeando colunas

In [ ]:
df_nintendo.rename(columns={'Package Id': 'Id Pacote',
                            'Game': 'Jogo',  
                            'Send Date': 'Data Envio',
                            'Region': 'Região',  
                            'Order Status': 'Status Pedido', 
                            'Stock Quantity': 'Quantidade Estoque', 
                            'Preço Unit Price': 'Preço Unitário',   
                            'valor_total_estoque': 'Valor Total Estoque',
                            'Store Manager': 'Gerente',  
                            'Age Rating': 'Faixa Etária'
                            }, inplace=True)

##### Reorganizando colunas

In [ ]:
minha_ordem = ['Id Pacote', 'Jogo', 'Console', 'Data Envio', 'Região', 'Status Pedido',
       'E-commerce', 'Quantidade Estoque', 'Preço Unitário', 'Valor Total Estoque', 'Gerente',
       'Batch', 'Faixa Etária', 'Publisher']

df_nintendo = df_nintendo[minha_ordem]

display(df_nintendo)

##### Dados Estatísticos da Tabela ETL

In [ ]:
df_nintendo.describe(include='number').round(2) #apenas colunas numéricas, arredondando os resultados em 2 casas decimais

##### Preenchendo valores nulos

In [ ]:
# 'Quantidade Estoque'
df_nintendo['Quantidade Estoque'] = df_nintendo['Quantidade Estoque'].fillna(0)

In [ ]:
# 'Preço Unitário' e 'Valor Total Estoque', que continuam sendo float
df_nintendo[['Preço Unitário','Valor Total Estoque']] = df_nintendo[['Preço Unitário','Valor Total Estoque']].fillna(0.00)

In [ ]:
df_nintendo[['Jogo','Região','Status Pedido','E-commerce','Gerente',  
             'Batch','Faixa Etária','Publisher']] = df_nintendo[['Jogo','Região','Status Pedido',  
                                                                 'E-commerce','Gerente','Batch','Faixa Etária',
                                                                 'Publisher']].fillna('Não disponível')

In [ ]:
df_nintendo.head(12)

In [ ]:
df_nintendo.info()

In [ ]:
# total de valores nulos em cada coluna
df_nintendo.isna().sum() 

##### Os ids preenchidos com traços passam a ser nulos

In [ ]:
df_nintendo['Id Pacote'] = df_nintendo['Id Pacote'].replace(['---','----','-----'], np.nan)

##### Substituindo os traços por "Não disponível" (nas colunas de texto)

In [ ]:
df_nintendo[['Jogo','Região','Status Pedido','E-commerce','Gerente',  
             'Batch','Faixa Etária','Publisher']] = df_nintendo[['Jogo','Região','Status Pedido',  
             'E-commerce','Gerente','Batch','Faixa Etária','Publisher']].replace('---','Não disponível', regex=False)

df_nintendo[['Jogo','Região','Status Pedido','E-commerce','Gerente',  
             'Batch','Faixa Etária','Publisher']] = df_nintendo[['Jogo','Região','Status Pedido',  
             'E-commerce','Gerente','Batch','Faixa Etária','Publisher']].replace('----','Não disponível', regex=False)

df_nintendo[['Jogo','Região','Status Pedido','E-commerce','Gerente',  
             'Batch','Faixa Etária','Publisher']] = df_nintendo[['Jogo','Região','Status Pedido',  
             'E-commerce','Gerente','Batch','Faixa Etária','Publisher']].replace('-----','Não disponível', regex=False)             

##### Lista de Jogos Nintendo

In [ ]:
jogos = df_nintendo['Jogo'].unique().tolist()
jogos.remove('Não disponível') #valor que não queremos listar

for jogo in jogos:
    print(jogo)

##### Substituindo os traços por 0 ou 0.00 (nas colunas numéricas)

In [ ]:
df_nintendo['Quantidade Estoque'] = df_nintendo['Quantidade Estoque'].replace(['---','----','-----'], 0, regex=False)

In [ ]:
df_nintendo[['Preço Unitário','Valor Total Estoque']] = df_nintendo[['Preço Unitário',  
                                                                     'Valor Total Estoque']].replace(['---','----','-----'], 
                                                                                                     0.00, regex=False)

In [ ]:
df_nintendo.info()

In [ ]:
df_nintendo.head(12) #as 12 primeiras linhas

In [ ]:
# print(df_nintendo['Jogo'].value_counts().to_string())
frequencia_jogos = df_nintendo['Jogo'].value_counts()
for jogo, cont in frequencia_jogos.items():
    print(f"{jogo} => {cont}")

In [ ]:
# df_nintendo['Região'].unique().tolist() => listar registros únicos (sem repetição)
frequencia_regioes = df_nintendo['Região'].value_counts()

for valor, cont in frequencia_regioes.items():
    print(f"{valor}: {cont}")

In [ ]:
frequencia_status_pedido = df_nintendo['Status Pedido'].value_counts()
for status, cont in frequencia_status_pedido.items():
    print(f"{status}: {cont}")

In [ ]:
frequencia_eCommerce = df_nintendo['E-commerce'].value_counts()
for comercio, cont in frequencia_eCommerce.items():
   print(f"{comercio}: {cont}") 

In [ ]:
frequencia_gerente = df_nintendo['Gerente'].value_counts()
for gerente, cont in frequencia_gerente.items():
    print(f"{gerente}: {cont}")

In [ ]:
frequencia_batch = df_nintendo['Batch'].value_counts()
for batch, cont in frequencia_batch.items():
    print(f"{batch}: {cont}")

In [ ]:
frequencia_faixaEtaria = df_nintendo['Faixa Etária'].value_counts()
for faixaEtaria, cont in frequencia_faixaEtaria.items():
    print(f"{faixaEtaria}: {cont}")

In [ ]:
frequencia_publisher = df_nintendo['Publisher'].value_counts()
for publisher, cont in frequencia_publisher.items():
    print(f"{publisher}: {cont}")

##### Exportando os dados limpos para um novo arquivo CSV

In [ ]:
df_nintendo.to_csv('..\\ETL\\jogosNintendo.csv', index=False, encoding='utf-8')

In [ ]:
df_nintendo.info()

##### "Data Envio" tem os mesmos 11 nulos de Id Pacote?<br>Vale conferir se são exatamente as mesmas linhas com valores ausentes nas duas colunas.

Para especificar quais colunas exibir no resultado, basta adicionar .loc (ou colchetes duplos) com a lista de colunas desejadas depois do filtro de linhas:

In [ ]:
df_nintendo.loc[
    df_nintendo['Id Pacote'].isna() | df_nintendo['Data Envio'].isna(),
    ['Id Pacote', 'Jogo', 'Console', 'Data Envio', 'Região', 'Status Pedido', 'Gerente']
]